# Activation Functions

Companion notebook for Section 3 of the MLP lecture notes.

Objectives:

- plot common hidden-layer activation functions;
- inspect their derivatives;
- see why nonlinear activations are necessary;
- visualize sigmoid/tanh saturation and vanishing gradients;
- understand the dying ReLU risk and the role of Leaky ReLU.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=6, suppress=True)


## 1. Definitions

We implement sigmoid, tanh, ReLU, and Leaky ReLU with their derivatives.


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def d_sigmoid(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh(z):
    return np.tanh(z)

def d_tanh(z):
    return 1 - np.tanh(z) ** 2

def relu(z):
    return np.maximum(0, z)

def d_relu(z):
    return (z > 0).astype(float)

def leaky_relu(z, alpha=0.1):
    return np.where(z > 0, z, alpha * z)

def d_leaky_relu(z, alpha=0.1):
    return np.where(z > 0, 1.0, alpha)


## 2. Activation curves

Sigmoid and tanh saturate at large positive or negative values. ReLU is zero on the negative side and linear on the positive side.


In [ ]:
z = np.linspace(-6, 6, 500)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(z, sigmoid(z), label='sigmoid')
ax.plot(z, tanh(z), label='tanh')
ax.plot(z, relu(z), label='ReLU')
ax.plot(z, leaky_relu(z), label='Leaky ReLU (alpha=0.1)')
ax.set_xlabel('z')
ax.set_ylabel('activation')
ax.set_title('Common activation functions')
ax.set_ylim(-1.2, 6.2)
ax.legend()
plt.show()


## 3. Derivatives

Backpropagation multiplies gradients by local derivatives. Small activation derivatives can make gradients shrink.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(z, d_sigmoid(z), label="sigmoid'")
ax.plot(z, d_tanh(z), label="tanh'")
ax.plot(z, d_relu(z), label="ReLU'")
ax.plot(z, d_leaky_relu(z), label="Leaky ReLU' (alpha=0.1)")
ax.set_xlabel('z')
ax.set_ylabel('derivative')
ax.set_title('Activation derivatives')
ax.legend()
plt.show()

print('Maximum sigmoid derivative:', d_sigmoid(np.array([0.0]))[0])
print('Sigmoid derivative at z=6:', d_sigmoid(np.array([6.0]))[0])
print('Sigmoid derivative at z=-6:', d_sigmoid(np.array([-6.0]))[0])


## 4. Why nonlinear activations are necessary

A stack of linear layers without nonlinear activations is still just one linear layer. The code below constructs two linear layers and then collapses them into one equivalent linear transformation.


In [ ]:
rng = np.random.default_rng(42)
W1 = rng.normal(size=(3, 2))
b1 = rng.normal(size=3)
W2 = rng.normal(size=(1, 3))
b2 = rng.normal(size=1)

x = rng.normal(size=2)

# Two linear layers.
h = W1 @ x + b1
y_two_layers = W2 @ h + b2

# Equivalent single linear layer.
W_equiv = W2 @ W1
b_equiv = W2 @ b1 + b2
y_single_layer = W_equiv @ x + b_equiv

print('Two linear layers:', y_two_layers)
print('Single equivalent layer:', y_single_layer)
print('Same output?', np.allclose(y_two_layers, y_single_layer))


## 5. Saturation and vanishing gradients

If sigmoid pre-activations are very large in magnitude, derivatives become small. In deep networks, repeatedly multiplying by small derivatives can make early-layer gradients nearly vanish.


In [ ]:
depths = np.arange(1, 21)
local_derivatives = [0.25, 0.10, 0.02, 0.005]

fig, ax = plt.subplots(figsize=(8, 5))
for d in local_derivatives:
    ax.semilogy(depths, d ** depths, marker='o', label=f'local derivative={d}')
ax.set_xlabel('Number of multiplied layers')
ax.set_ylabel('Gradient scale factor')
ax.set_title('Repeated small derivatives shrink gradients exponentially')
ax.legend()
plt.show()


## 6. Sigmoid versus ReLU gradient flow in a toy deep chain

This is not a full network. It isolates the effect of multiplying activation derivatives along a deep path.


In [ ]:
rng = np.random.default_rng(0)
n_paths = 5000
max_depth = 30

# Pre-activations with moderate spread. Sigmoid often sees derivatives below 0.25.
z_samples = rng.normal(loc=0.0, scale=2.5, size=(n_paths, max_depth))

sigmoid_factors = np.cumprod(d_sigmoid(z_samples), axis=1)
relu_factors = np.cumprod(d_relu(z_samples), axis=1)
leaky_factors = np.cumprod(d_leaky_relu(z_samples, alpha=0.1), axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(np.arange(1, max_depth + 1), np.median(sigmoid_factors, axis=0), label='median sigmoid path')
ax.semilogy(np.arange(1, max_depth + 1), np.mean(relu_factors, axis=0), label='mean ReLU path active')
ax.semilogy(np.arange(1, max_depth + 1), np.median(leaky_factors, axis=0), label='median Leaky ReLU path')
ax.set_xlabel('Depth')
ax.set_ylabel('Gradient multiplier')
ax.set_title('Toy gradient multipliers through activation derivatives')
ax.legend()
plt.show()


## 7. Dying ReLU and Leaky ReLU

A ReLU neuron receiving only negative pre-activations outputs zero and has zero derivative. Leaky ReLU keeps a small gradient on the negative side.


In [ ]:
z_negative = np.linspace(-5, -0.1, 10)
relu_table = pd.DataFrame({
    'z': z_negative,
    'relu(z)': relu(z_negative),
    "relu'(z)": d_relu(z_negative),
    'leaky_relu(z)': leaky_relu(z_negative),
    "leaky_relu'(z)": d_leaky_relu(z_negative),
})
relu_table


## 8. Hidden-layer versus output-layer activations

For hidden layers, ReLU or variants are strong defaults. For output layers, the activation must match the task:

- regression: linear output;
- binary classification: sigmoid probability or raw logit with a logits-aware loss;
- multi-class classification: softmax probabilities or raw logits with a logits-aware loss.


## Exercises

1. Change the scale of `z_samples` from 2.5 to 0.5 and 6.0. What happens to sigmoid gradient flow?
2. Change the Leaky ReLU alpha from 0.1 to 0.01. What changes?
3. Why is ReLU not appropriate as the final activation for a binary classifier probability?


## Takeaways

- Nonlinear activations are what make depth useful.
- Sigmoid and tanh can saturate and produce very small gradients.
- ReLU improves gradient flow for positive pre-activations, but can produce inactive neurons.
- Output activations are task-dependent; hidden-layer defaults do not automatically apply to outputs.
